In [ ]:
import pandas as pd
import numpy as np


def clean_and_prep(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Junk Filter: Remove test/dummy data
    junk_keywords = ["test", "testing", "dummy", "sample", "delete"]
    mask_junk = df['consumer_complaint_narrative'].str.lower().apply(lambda x: any(k in x for k in junk_keywords))
    df = df[~mask_junk]

    # 2. Impute Metadata: Handle missing values for search stability
    impute_map = {
        'state': 'Unknown',
        'company_public_response': 'No public response',
        'company_response_to_consumer': 'No response recorded',
        'submitted_via': 'Web'  # Default
    }
    for col, val in impute_map.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

    # 3. Intelligence Engineering: Calculate response speed
    # Convert dates to datetime, coerce errors to NaT
    df['date_received'] = pd.to_datetime(df['date_received'], errors='coerce')
    df['date_sent_to_company'] = pd.to_datetime(df['date_sent_to_company'], errors='coerce')
    df['days_to_respond'] = (df['date_sent_to_company'] - df['date_received']).dt.days.fillna(0).astype(int)

    return df


def build_full_context(row):
    return (
        f"Product: {row['product']} ({row['sub_product']})\n"
        f"Company: {row['company']} | State: {row['state']}\n"
        f"Submitted via: {row['submitted_via']} | Resolution: {row['company_response_to_consumer']}\n"
        f"Response Time: {row['days_to_respond']} days\n"
        f"Complaint Narrative: {row['consumer_complaint_narrative']}"
    )


def main():
    df = pd.read_parquet("raw_complaints.parquet")
    df = clean_and_prep(df)
    df['full_context'] = df.apply(build_full_context, axis=1)

    # Verification
    print(f"Processed {len(df)} records for embedding.")
    print(f"Sample Context:\n{df['full_context'].iloc[0]}")

    df.to_parquet("processed_complaints.parquet", compression="snappy")


In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("complaints_sample_2000.csv")

df.describe()          # summary stats for numeric columns (count, mean, std, min, max, quartiles)
df.describe(include="object")   # same, but for text/categorical columns (count, unique, top, freq)
df.info()              # column names, dtypes, non-null counts — great first check
df.shape                # (rows, columns)
df.isnull().sum()       # missing values per column
df["Product"].value_counts()    # frequency counts for a specific column
df.dtypes               # data type of each column

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Date received                 2000 non-null   str    
 1   Product                       2000 non-null   str    
 2   Sub-product                   2000 non-null   str    
 3   Issue                         2000 non-null   str    
 4   Sub-issue                     1919 non-null   str    
 5   Consumer complaint narrative  193 non-null    str    
 6   Company public response       824 non-null    str    
 7   Company                       2000 non-null   str    
 8   State                         1994 non-null   str    
 9   ZIP code                      1998 non-null   str    
 10  Tags                          81 non-null     str    
 11  Submitted via                 2000 non-null   str    
 12  Date sent to company          1939 non-null   str    
 13  Company respon

/var/folders/qw/xcms2yw12tv6zypctqzl5b7m0000gn/T/ipykernel_16153/2685287057.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")   # same, but for text/categorical columns (count, unique, top, freq)


Date received                       str
Product                             str
Sub-product                         str
Issue                               str
Sub-issue                           str
Consumer complaint narrative        str
Company public response             str
Company                             str
State                               str
ZIP code                            str
Tags                                str
Submitted via                       str
Date sent to company                str
Company response to consumer        str
Timely response?                    str
Complaint ID                    float64
dtype: object

In [3]:
df["Product"].value_counts().head(10)        # most common complaint categories
df["Company"].value_counts().head(10)        # most complained-about companies
df["State"].value_counts().head(10)          # complaints by state
df["Date received"] = pd.to_datetime(df["Date received"])
df["Date received"].dt.year.value_counts()   # complaints by year

Date received
2026    1859
2025     138
2024       3
Name: count, dtype: int64